# DCS Paddy Supply Preprocessing

Converts seasonal paddy statistics into daily supply features.

In [ ]:
import re
import pandas as pd
from paths import INTEGRATION_END_DATE, PREPROCESSED_PATHS, RAW_PATHS, save_dataset


In [ ]:
def assign_market_date(season_str):
    if 'Yala' in season_str:
        match = re.search(r'(\d{4})Yala', season_str, re.IGNORECASE)
        if match:
            return pd.to_datetime(f'{match.group(1)}-09-01')
    elif 'Maha' in season_str:
        match = re.search(r'(\d{4})Maha', season_str, re.IGNORECASE)
        if match:
            return pd.to_datetime(f'{match.group(1)}-03-01')
    return pd.NaT

df_dcs = pd.read_csv(RAW_PATHS['dcs_paddy'])
df_national = df_dcs.groupby('Season').agg(
    Total_Production_MT=('Total_Production_MT', 'sum'),
    Gross_Harvested_Total=('Gross_Harvested_Total', 'sum'),
).reset_index()
df_national['National_Yield_Efficiency'] = (
    df_national['Total_Production_MT'] / df_national['Gross_Harvested_Total']
)
df_national['Market_Date'] = df_national['Season'].apply(assign_market_date)
df_national = df_national.dropna(subset=['Market_Date']).sort_values('Market_Date').set_index('Market_Date')

daily_idx = pd.date_range(start=df_national.index.min(), end=pd.to_datetime(INTEGRATION_END_DATE), freq='D')
df_daily = pd.DataFrame(index=daily_idx)
df_daily.index.name = 'Report_Date'
df_daily = df_daily.join(df_national[['Total_Production_MT', 'National_Yield_Efficiency', 'Season']]).ffill()
df_daily['Supply_YoY_Change_%'] = df_daily['Total_Production_MT'].pct_change(periods=365) * 100

save_dataset(df_daily, PREPROCESSED_PATHS['dcs_daily'])
print('DCS daily shape:', df_daily.shape)
